In [1]:
# from langchain_ollama import ChatOllama
# from langchain_core.tools import tool

# llm = ChatOllama(
#     model="llama3.1:8b"
# )

# response = llm.invoke("What is the capital of France?")
# print(response.content)

In [2]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.tools import tool
from dotenv import load_dotenv
import os


load_dotenv()

os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")


In [3]:
llm = ChatGoogleGenerativeAI(
    model="gemini-flash-latest"
)
llm

ChatGoogleGenerativeAI(profile={'max_input_tokens': 1048576, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': True, 'pdf_inputs': True, 'video_inputs': True, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'image_url_inputs': True, 'image_tool_message': True, 'tool_choice': True}, google_api_key=SecretStr('**********'), model='gemini-flash-latest', client=<google.genai.client.Client object at 0x0000028F236B1FF0>, default_metadata=(), model_kwargs={})

In [4]:
llm.invoke("What is the capital of France?")

AIMessage(content=[{'type': 'text', 'text': 'The capital of France is **Paris**.', 'extras': {'signature': 'Eq8BCqwBAb4+9vvUYMSksNB5VwxVhwEca4LMnrg9xjxh63HtjUhZyjIhLqd26GlJPR/RAJpYBgBS+EGSXA3EhZu+dyjEI6hfOqE5fqdwklQ6YhpyZdOeQ+z4lnZE881/Du8kvWZhm2Tb+H8ss4l4UVw7eSesq2E7uOJ8WcHfgHWE3dLdPdcJFBGBXbdeYA+ldEFQugEmNLHEMQD9lYKnXXUcWO+kQGUhNY6LxSoQOPnNKA=='}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3-flash-preview', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019d5c03-d3c5-7031-8029-f242bef4b980-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 8, 'output_tokens': 40, 'total_tokens': 48, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 32}})

In [5]:
from langchain_community.tools import DuckDuckGoSearchRun
@tool
def duck_search(query: str):
    """Use this tool to search for information on the web using DuckDuckGo."""
    search_tool = DuckDuckGoSearchRun()
    return search_tool.invoke(query)

In [6]:
from langchain_community.retrievers import ArxivRetriever

arxiv_retriever = ArxivRetriever(
    load_max_docs=2,
    get_full_documents=True
)

In [7]:
arxiv_retriever.invoke("Transformer architecture")

[Document(metadata={'Published': '2021-12-24', 'Title': 'Architectural Implications of Graph Neural Networks', 'Authors': 'Zhihui Zhang, Jingwen Leng, Lingxiao Ma, Youshan Miao, Chao Li, Minyi Guo', 'Summary': 'Graph neural networks (GNN) represent an emerging line of deep learning models that operate on graph structures. It is becoming more and more popular due to its high accuracy achieved in many graph-related tasks. However, GNN is not as well understood in the system and architecture community as its counterparts such as multi-layer perceptrons and convolutional neural networks. This work tries to introduce the GNN to our community. In contrast to prior work that only presents characterizations of GCNs, our work covers a large portion of the varieties for GNN workloads based on a general GNN description framework. By constructing the models on top of two widely-used libraries, we characterize the GNN computation at inference stage concerning general-purpose and application-specifi

In [8]:
from langchain_community.tools import ArxivQueryRun
from langchain_community.utilities import ArxivAPIWrapper
from sympy import deg

@tool
def arxiv_tool(name: str):
    """Use this tool to get information from Arxiv Database for research papers."""
    arxiv_query = ArxivQueryRun(api_wrapper=ArxivAPIWrapper())
    return arxiv_query.invoke(name)

In [9]:
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

@tool
def wiki_tool(name: str):
    """Use this tool to get information about a person from Wikipedia."""
    wiki_query = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper())
    return wiki_query.invoke(name)


In [10]:
#Custom Tool
@tool
def personal_info(name: str):
    """Use this tool to get personal information about a person."""
    info = {
        "Virat Kohli": "Virat Kohli is an Indian cricketer and former captain of the Indian national team.",
        "Sachin Tendulkar": "Sachin Tendulkar is a former Indian cricketer and one of the greatest batsmen in the history of cricket."
    }
    return info.get(name, "Information not available.")

In [11]:
#Tool Binding
tools = [duck_search, arxiv_tool, wiki_tool, personal_info]

llm_with_tools = llm.bind_tools(tools)

In [12]:
llm_with_tools.invoke("Who is Virat Kohli?")

AIMessage(content=[], additional_kwargs={'function_call': {'name': 'wiki_tool', 'arguments': '{"name": "Virat Kohli"}'}, '__gemini_function_call_thought_signatures__': {'50c922c5-8d0a-4900-b6e7-e510ae2ec28f': 'EqcCCqQCAb4+9vtjLG/MCtftLtZ5LoF4W0FEh5K2CPn3BRN/fXffHr7b0Pcv8FHtXZIxXkBi5+VYvO2xhncEsoKF8k2GQkK16JcgUleL6x5UsL70fR35scfmpvMtNItD876gq3Qf8HmL2NOzmegwy+I5/Rra4alsfipxVBtEbrboO0ZrZXomTcwuSxt7GzYM0kg3IoTtDMB4cFLF5hKob81y63/3YNHAad5e2aKi9wBxS9m5VkW53Zp6hQSadnt+6Oa++OFYwSTDTXOju3nLGBbakawR+QAfqm+7gCffOKbxjzu1FKmUKD3Uv0zU9IUF211+XbgZArz8CWKJapzJeaA/7VULZTMXmfzsB1kQEnpiquLGqESb/QELgSlCKUpvsHdJdeUcL23DYg=='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3-flash-preview', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019d5c04-80d7-7d70-94c1-38ae055a1aca-0', tool_calls=[{'name': 'wiki_tool', 'args': {'name': 'Virat Kohli'}, 'id': '50c922c5-8d0a-4900-b6e7-e510ae2ec28f', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 